## Charged tensor operations

A symmetric MPS is a network of intertwiners: every tensor is invariant, and the charge of the
state is carried by the bonds.  An operator that carries a charge of its own -- the triplet part
of a bond operator, built here as an MPO with `total=(2,)` -- moves the state into another
sector, so the bonds of the result have to carry the charge it adds.

The blocks of a tensor store one number per multiplet, the fusion path is their label, and the
physical components live in the Clebsch-Gordan tensors of the tree.  A contraction therefore
never materialises a dense array: it multiplies the reduced blocks and weighs every output tree
by its recoupling factor.

In [ ]:
import torch

from tentorch.config import COMPLEX, CUDA
from tentorch import Broomstick, MPO, MPS
from tentorch.symmetry import gell_mann, su2_symmetry, su3_symmetry
from tentorch.tensors import Tensor

LENGTH, PHYS = 6, 2


def bond_table(state: MPS) -> None:
    '''
    Print the sectors, their multiplicities, and the number of states on every bond.

    Args:
        state: the state whose bond legs are read, the left edge included.
    '''
    print(f'{"bond":>5}{"multiplets":>12}{"states":>8}   blocks')
    for index, tensor in enumerate(state.tensors):
        leg = tensor.legs[2]
        blocks = '  '.join(f'{leg.sector(block)} x{leg.multiplicity(block)}'
                           for block in range(leg.nblocks))
        print(f'{index + 1:>5}{leg.multiplets:>12}{leg.dim:>8}   {blocks}')


stick = Broomstick(L=LENGTH, phys_dim=PHYS, max_bond_dim=32, svd_tol=1e-14,
                   group=su2_symmetry, site_sector=(1,))
for site in range(LENGTH - 1):
    for axis in 'XYZ':
        stick.add_couplings((axis, axis), 1.0, (site, site + 1))
stick.build()
stick.sweep(num_sweeps=6)
state = stick.state

print(f'Heisenberg L={LENGTH}: energy {stick.energy:.12f}, per bond {stick.energy / LENGTH:.12f}')
bond_table(state)
print('bond_dim (multiplets)', state.bond_dim)

The bond operator $\vec S_i \cdot \vec S_{i+1}$ of the Heisenberg chain is invariant, so every MPO
tensor of the Hamiltonian is neutral on all of its legs.  Placing the same three couplings in the
charge channel `total=(2,)` builds an operator whose bonds carry the triplet charge instead.

The reference is analytic rather than dense, because a dense array is not an object of a
symmetric network: $S_0\cdot S_1 = \tfrac14 P_t - \tfrac34 P_s$, so the triplet weight is
$\langle P_t\rangle = \langle S_0\cdot S_1\rangle + \tfrac34$ and
$\langle (S_0\cdot S_1)^2\rangle = \tfrac{9}{16} - \tfrac12\langle P_t\rangle$.  The engine
applies the MPO to the MPS and the bonds of the result carry the charge the operator added.

In [ ]:
def bond_mpo(total: tuple[int, ...] | None = None) -> MPO:
    '''
    The Heisenberg bond operator on sites (0, 1), in one charge channel.

    Args:
        total: the total charge the MPO carries; None for the invariant operator.
    Returns:
        The MPO with the three Heisenberg couplings on the bond (0, 1).
    '''
    operator = MPO(L=LENGTH, phys_dim=PHYS, group=su2_symmetry, total=total, site_sector=(1,))
    for axis in 'XYZ':
        operator.add_couplings((axis, axis), 1.0, (0, 1))
    operator.build()
    return operator


invariant, charge = bond_mpo(), bond_mpo((2,))
bond = float(state.expectation(invariant).real)
triplet = bond + 0.75
reference = 0.5625 - 0.5 * triplet
applied = state.apply(charge, max_bond_dim=32, tol=1e-14)

print(f'{"":<14}{"|O psi|^2":>22}{"|diff|":>12}')
print(f'{"analytic":<14}{reference:>22.15f}')
print(f'{"applied":<14}{float(applied.norm().real):>22.15f}'
      f'{abs(float(applied.norm().real) - reference):>12.2e}')
print(f'<S.S> {bond:.12f}   <P_t> {triplet:.12f}')
bond_table(applied)
print('bond_dim (multiplets)', applied.bond_dim)

The charge of the operator lands on the bonds of the result: every bond sector of the applied
state is the fusion of the state's own charge with the operator's $(2,)$, so the bond table shows
$(0,)$ next to $(2,)$ and $(1,)$ next to $(3,)$, and the number of multiplets grows accordingly.
The state has left the trivial sector it started in.

In [ ]:
DIM3 = 3

generators = gell_mann(dtype=COMPLEX, device=CUDA)
mapping = {f'L{index + 1}': Tensor.from_torch(generator)
           for index, generator in enumerate(generators)}
mapping['I'] = Tensor.from_torch(torch.eye(DIM3, dtype=COMPLEX, device=CUDA))

stick3 = Broomstick(L=LENGTH, phys_dim=DIM3, max_bond_dim=32, svd_tol=1e-14,
                    mapping=mapping, group=su3_symmetry, site_sector=(1, 0))
for site in range(LENGTH - 1):
    stick3.add_couplings(('I', 'I'), 1.0 / 3.0, (site, site + 1))
    for name in mapping:
        if name != 'I':
            stick3.add_couplings((name, name), 0.5, (site, site + 1))
stick3.build()
stick3.sweep(num_sweeps=6)
state3 = stick3.state

adjoint = MPO(L=LENGTH, phys_dim=DIM3, mapping=mapping, group=su3_symmetry,
              total=(1, 1), site_sector=(1, 0))
adjoint.add_couplings(('L1', 'L1'), 1.0, (0, 1))
adjoint.build()
applied3 = state3.apply(adjoint, max_bond_dim=32, tol=1e-14)

print(f'ULS L={LENGTH}: energy {stick3.energy:.12f}, per bond {stick3.energy / LENGTH:.12f}')
print(f'adjoint channel applied   |O psi|^2 {float(applied3.norm().real):.15f}')
bond_table(applied3)

An operator enters a symmetric network as an MPO, whose bond legs carry the charge of
every channel.  `apply` takes the operator in that form and refolds the result to the
bond dimension it is given.


In [ ]:
triplet_channel = bond_mpo((2,))
applied_triplet = state.apply(triplet_channel, max_bond_dim=32)
print('the triplet channel as an MPO   |O psi|^2',
      f'{float(applied_triplet.norm().real):.15f}')
